In [ ]:
# data/load_data.py
import pandas as pd
import numpy as np
import os
import textstat

def load_and_clean():

    # Create output folder 
    os.makedirs("data", exist_ok=True)


    books   = pd.read_csv("Books.csv",   encoding="latin-1", low_memory=False)
    ratings = pd.read_csv("Ratings.csv", encoding="latin-1")
    users   = pd.read_csv("Users.csv",   encoding="latin-1")

   
    print("RAW DATA SHAPES")
    print(f"  Books:   {books.shape}")
    print(f"  Ratings: {ratings.shape}")
    print(f"  Users:   {users.shape}")

    
    # BOOKS CLEANING
    
    # Keep only useful columns
    books = books[["ISBN", "Book-Title", "Book-Author",
                   "Year-Of-Publication", "Publisher", "Image-URL-M"]]

    # Fix bad publication years
    books["Year-Of-Publication"] = pd.to_numeric(
        books["Year-Of-Publication"], errors="coerce")
    books = books[books["Year-Of-Publication"].between(1800, 2024)]

    # Filling missing values
    books["Book-Author"] = books["Book-Author"].fillna("Unknown")
    books["Publisher"]   = books["Publisher"].fillna("Unknown")
    books["Image-URL-M"] = books["Image-URL-M"].fillna("")

    # Clean title
    # .str.strip() removes leading and trailing whitespace from every title. Without this, "Harry Potter " and "Harry Potter" would be treated as two different books because one has a trailing space.
    books["Book-Title"] = books["Book-Title"].str.strip()

    # Drop duplicates
    books = books.drop_duplicates(subset="ISBN").reset_index(drop=True)

    print(f"\nBooks after cleaning: {books.shape}")

    # RATINGS CLEANING
    
    # Drop implicit ratings (0 = interacted but never rated)
    ratings = ratings[ratings["Book-Rating"] != 0]
    print(f"Ratings after removing zeros: {ratings.shape}")

    # Keep only ratings for books that exist in Books.csv
    ratings = ratings[ratings["ISBN"].isin(books["ISBN"])].reset_index(drop=True)
    print(f"Ratings after ISBN filter: {ratings.shape}")

    
    # USERS CLEANING
   

    users["Age"] = pd.to_numeric(users["Age"], errors="coerce")
    users["Age"] = users["Age"].apply(
        lambda x: x if pd.notna(x) and 5 <= x <= 90 else np.nan)
    users["Age"] = users["Age"].fillna(users["Age"].median())

    
    # UNDERSTAND  DATA BEFORE FILTERING
   


    print("THRESHOLD ANALYSIS")
    

    user_counts = ratings["User-ID"].value_counts()
    book_counts = ratings["ISBN"].value_counts()

    print(f"Users with 200+ ratings : {(user_counts >= 200).sum()}")
    print(f"Users with 100+ ratings : {(user_counts >= 100).sum()}")
    print(f"Users with  50+ ratings : {(user_counts >=  50).sum()}")
    print(f"Users with  20+ ratings : {(user_counts >=  20).sum()}")
    print(f"\nBooks with  50+ ratings : {(book_counts >=  50).sum()}")
    print(f"Books with  20+ ratings : {(book_counts >=  20).sum()}")
    print(f"Books with  10+ ratings : {(book_counts >=  10).sum()}")

    
    # FILTER FOR CF — ACTIVE USERS + POPULAR BOOKS
    # Step 1 — filter active users (rated 50+ books)
    active_users = (user_counts[user_counts >= 50]).index
    ratings_cf   = ratings[ratings["User-ID"].isin(active_users)]
    print(f"\nAfter user filter  (>=50 ratings) : {ratings_cf.shape}")
    print(f"  Unique users : {ratings_cf['User-ID'].nunique()}")

    # Step 2 — filter popular books (rated by 20+ users)
    book_counts_cf = ratings_cf["ISBN"].value_counts()
    popular_books  = (book_counts_cf[book_counts_cf >= 20]).index
    ratings_cf     = ratings_cf[ratings_cf["ISBN"].isin(popular_books)]
    print(f"\nAfter book filter  (>=20 ratings) : {ratings_cf.shape}")
    print(f"  Unique users : {ratings_cf['User-ID'].nunique()}")
    print(f"  Unique books : {ratings_cf['ISBN'].nunique()}")

    # Safety check
    if ratings_cf.empty or ratings_cf["User-ID"].nunique() == 0 \
                        or ratings_cf["ISBN"].nunique() == 0:
        print("\nERROR: ratings_cf is still empty.")
        print("Lower thresholds further or check your CSV files.")
        return None, None, None, None

    # Merge with book metadata
    ratings_cf = ratings_cf.merge(books, on="ISBN", how="left")

    # Sparsity
    n_users  = ratings_cf["User-ID"].nunique()
    n_books  = ratings_cf["ISBN"].nunique()
    sparsity = 1 - len(ratings_cf) / (n_users * n_books)

    print("\n" + "=" * 50)
    print("FINAL CF DATASET")
    print("=" * 50)
    print(f"  Rows     : {len(ratings_cf)}")
    print(f"  Users    : {n_users}")
    print(f"  Books    : {n_books}")
    print(f"  Sparsity : {sparsity:.4f}  ({sparsity*100:.1f}% empty)")

    # SAVE ALL CLEANED FILES


    books.to_csv("data/books_clean.csv",       index=False)
    ratings.to_csv("data/ratings_clean.csv",   index=False)
    ratings_cf.to_csv("data/ratings_cf.csv",   index=False)
    users.to_csv("data/users_clean.csv",       index=False)

  
    print("ALL FILES SAVED TO data/")
    print("  data/books_clean.csv")
    print("  data/ratings_clean.csv")
    print("  data/ratings_cf.csv")
    print("  data/users_clean.csv")

    return books, ratings, ratings_cf, users


if __name__ == "__main__":
    books, ratings, ratings_cf, users = load_and_clean()

    if books is not None:
        print("\nSample books_clean:")
        print(books[["Book-Title","Book-Author",
                      "reading_difficulty"]].head(3))

        print("\nSample ratings_cf:")
        print(ratings_cf[["User-ID","ISBN",
                           "Book-Rating","Book-Title"]].head(3))

RAW DATA SHAPES
  Books:   (271360, 8)
  Ratings: (1149780, 3)
  Users:   (278858, 3)

Books after cleaning: (266725, 6)
Ratings after removing zeros: (433671, 3)
Ratings after ISBN filter: (378032, 3)

THRESHOLD ANALYSIS
Users with 200+ ratings : 116
Users with 100+ ratings : 438
Users with  50+ ratings : 1163
Users with  20+ ratings : 3253

Books with  50+ ratings : 529
Books with  20+ ratings : 2112
Books with  10+ ratings : 5381

After user filter  (>=50 ratings) : (150970, 3)
  Unique users : 1163

After book filter  (>=20 ratings) : (11149, 3)
  Unique users : 1084
  Unique books : 349

FINAL CF DATASET
  Rows     : 11149
  Users    : 1084
  Books    : 349
  Sparsity : 0.9705  (97.1% empty)

Computing reading difficulty scores...
  Easy   : 30708
  Medium : 93372
  Hard   : 142645

ALL FILES SAVED TO data/
  data/books_clean.csv
  data/ratings_clean.csv
  data/ratings_cf.csv
  data/users_clean.csv

Sample books_clean:
             Book-Title           Book-Author reading_difficul